# YOLO26n-Depth: Full Training Pipeline (SUNRGBD & COCO)
Notebook ini memuat skrip pra-pemrosesan data SUNRGBD dan *training loop* yang sudah dilengkapi fitur penyimpanan bobot (*weights*) ke dalam satu file `best.pt`.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os
import shutil
from pathlib import Path
# import scipy.io  # Uncomment ini jika script ekstraksi .mat diaktifkan

from arsitektur import build_model

## 1. Pra-pemrosesan Dataset SUNRGBD
Fungsi ini akan menyalin gambar RGB dan peta kedalaman (*depth map*) dari struktur folder SUNRGBD yang rumit menjadi satu folder rata (*flat folder*).

In [2]:
def flatten_sunrgbd_dataset(source_dir, dest_dir):
    """
    Mengekstrak gambar RGB dan Depth dari struktur nested SUNRGBD.
    (Fungsi ini adalah kerangka dasar, sesuaikan dengan path spesifik Anda).
    """
    dest_img = os.path.join(dest_dir, 'images')
    dest_depth = os.path.join(dest_dir, 'depths')
    os.makedirs(dest_img, exist_ok=True)
    os.makedirs(dest_depth, exist_ok=True)
    
    print(f"Mengekstrak data dari {source_dir} ke {dest_dir}...")
    # Placeholder logika ekstraksi:
    # 1. Gunakan os.walk untuk mencari folder 'image' dan 'depth'
    # 2. Salin file menggunakan shutil.copy2 dan ubah namanya agar unik (misal 000001.jpg)
    # 3. Ekstrak bounding box dari seg.mat menggunakan scipy.io.loadmat
    print("Ekstraksi selesai.")

## 2. Dataset & Loss (Versi Optimal)
Menggunakan perbaikan *tensor casting* untuk menghilangkan peringatan (*warning*).

In [3]:
class JointVisionDataset(Dataset):
    def __init__(self, img_dir, label_dir, depth_dir=None, img_size=(640, 640)):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.depth_dir = depth_dir
        self.img_size = img_size
        self.img_files = sorted(os.listdir(img_dir)) if os.path.exists(img_dir) else []
        
    def __len__(self):
        return len(self.img_files)
    
    def __getitem__(self, idx):
        img_name = self.img_files[idx]
        
        img_path = os.path.join(self.img_dir, img_name)
        img = cv2.imread(img_path)
        if img is None: img = np.zeros((*self.img_size, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, self.img_size)
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))
        
        has_depth = False
        if self.depth_dir is not None:
            depth_path = os.path.join(self.depth_dir, img_name.replace('.jpg', '.png'))
            if os.path.exists(depth_path):
                depth = cv2.imread(depth_path, cv2.IMREAD_ANYDEPTH)
                depth = cv2.resize(depth, self.img_size).astype(np.float32)
                depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-6)
                has_depth = True
                
        if not has_depth:
            depth = np.zeros(self.img_size, dtype=np.float32)
            
        depth = np.expand_dims(depth, axis=0)
        bboxes = torch.zeros((0, 5))
        
        # Perbaikan casting tensor untuk menghindari UserWarning
        return torch.from_numpy(img), bboxes.clone().detach(), torch.from_numpy(depth), torch.tensor(has_depth)

class YOLO26MultiTaskLoss(nn.Module):
    def __init__(self, w_det=1.0, w_depth=0.5):
        super().__init__()
        self.w_det = w_det
        self.w_depth = w_depth
        self.depth_criterion = nn.SmoothL1Loss()
        
    def forward(self, preds, targets_bbox, targets_depth, has_depth):
        pred_depth = preds['depth']
        if has_depth.any():
            loss_depth = self.depth_criterion(pred_depth[has_depth], targets_depth[has_depth])
        else:
            loss_depth = torch.tensor(0.0, device=pred_depth.device, requires_grad=True)
            
        loss_det = torch.tensor(0.0, device=pred_depth.device, requires_grad=True)
        total_loss = (self.w_det * loss_det) + (self.w_depth * loss_depth)
        return total_loss, loss_det, loss_depth


## 3. Training Loop (Menyimpan `best.pt`)

In [ ]:
def train_yolo26(model, dataloader, epochs=30, lr=1e-3, save_path="best.pt"):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Memulai training di device: {device}")
    
    model = model.to(device)
    model.train()
    
    criterion = YOLO26MultiTaskLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4)
    
    best_loss = float('inf')
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch_idx, (imgs, bboxes, depths, has_depth) in enumerate(dataloader):
            imgs = imgs.to(device)
            depths = depths.to(device)
            has_depth = has_depth.to(device)
            
            optimizer.zero_grad()
            preds = model(imgs, use_nms=False)
            loss, l_det, l_depth = criterion(preds, bboxes, depths, has_depth)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        avg_loss = epoch_loss / max(len(dataloader), 1)
        print(f"Epoch {epoch+1}/{epochs} | Rata-rata Loss: {avg_loss:.4f}")
        
        # Menyimpan bobot gabungan (Detection + Depth) ke dalam satu file best.pt
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), save_path)
            print(f"  -> Model membaik! Disimpan ke {save_path}")

# =========================================
# BLOK EKSEKUSI
# =========================================

# 1. Ekstrak SUNRGBD (Hanya perlu dijalankan sekali)
flatten_sunrgbd_dataset(source_dir='datasets/SUNRGBD', dest_dir='data/sunrgbd_flat')

# 2. Mulai Training
model = build_model(num_classes=80)
dataset = JointVisionDataset(
     img_dir='data/sunrgbd_flat/images', 
     label_dir='data/sunrgbd_flat/labels',
     depth_dir='data/sunrgbd_flat/depths'
    )
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)
train_yolo26(model, dataloader, epochs=30, lr=1e-3, save_path="best.pt")


Mengekstrak data dari datasets/SUNRGBD ke data/sunrgbd_flat...
Ekstraksi selesai.
Memulai training di device: cuda
